In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip install -q keras-hub
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))     # expect 1 GPU


Mounted at /content/drive
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import sys, json
ROOT = "/content/drive/MyDrive/ticket_routing"
sys.path.append(ROOT)

import numpy as np, pandas as pd, tensorflow as tf, keras, keras_hub
from src.text import clean_text, format_for_bert

keras.mixed_precision.set_global_policy("mixed_float16")     # ⚡ BEFORE building the model

label_order = json.load(open(f"{ROOT}/models/label_order.json"))
label_to_id = {l: i for i, l in enumerate(label_order)}

def first_350(text):
    return " ".join(text.split()[:350])

data = {}
for name in ["train", "val"]:
    df = pd.read_parquet(f"{ROOT}/data/processed/{name}.parquet")
    df["text_bert"] = df["Consumer complaint narrative"].map(clean_text).map(first_350).map(format_for_bert)
    df["y"] = df["label"].map(label_to_id)
    data[name] = df
    print(name, df.shape, "| NaN labels:", df["y"].isna().sum())     # expect 0


ModuleNotFoundError: No module named 'src'

In [ ]:
preprocessor = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
    "bert_small_en_uncased", sequence_length=512)
model = keras_hub.models.BertTextClassifier.from_preset(
    "bert_small_en_uncased", num_classes=11, preprocessor=preprocessor)
model.summary()


Preprocessor: "bert_text_classifier_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ bert_tokenizer (BertTokenizer)                                │                       Vocab size: 30,522 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "bert_text_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ segment_ids (InputLayer)      │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bert_backbone (BertBackbone)  │ [(None, 512), (None,      │      28,763,648 │ padding_mask[0][0],        │
│                               │ None, 512)]               │                 │ segment_ids[0][0],         │
│                               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ classifier_dropout (Dropout)  │ (None, 512)               │               0 │ bert_backbone[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ logits (Dense)                │ (None, 11)                │           5,643 │ classifier_dropout[0][0]   │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 28,769,291 (109.75 MB)

 Trainable params: 28,769,291 (109.75 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),     # 0.00005, the fine-tuning rate
    metrics=["accuracy"],
)


In [ ]:
val_ds = (tf.data.Dataset.from_tensor_slices((data["val"]["text_bert"].values, data["val"]["y"].values))
          .batch(32).prefetch(tf.data.AUTOTUNE))          # no shuffle!
model.evaluate(val_ds.take(1))                             # expect loss ≈ 2.40


1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.1562 - loss: 2.3271


[2.3271007537841797, 0.15625]

In [ ]:
import math, time

train = data["train"].sample(frac=1, random_state=42)          # undo the duplicate-count sort
share = (train["label"].iloc[:20_000] == "Credit reporting or other personal consumer reports").mean()
print(f"First 20k credit reporting share: {share:.1%}")         # expect ≈ 69%, NOT 96%

train_ds = (tf.data.Dataset.from_tensor_slices((train["text_bert"].values, train["y"].values))
            .shuffle(20_000, seed=42)
            .batch(32)
            .prefetch(tf.data.AUTOTUNE))


First 20k credit reporting share: 69.0%


In [ ]:
fit_times = []
for e in range(2):
    s = time.perf_counter()
    h = model.fit(train_ds.take(50), epochs=1)                  # epochs=1! (Day 10 bug 😉)
    fit_times.append(time.perf_counter() - s)
    print(f"Run {e + 1}: {fit_times[-1]:.1f} s | loss {h.history['loss'][0]:.4f}")


50/50 ━━━━━━━━━━━━━━━━━━━━ 39s 156ms/step - accuracy: 0.6731 - loss: 1.2225
Run 1: 39.8 s | loss 1.2225
50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 156ms/step - accuracy: 0.7525 - loss: 0.8331
Run 2: 8.8 s | loss 0.8331


In [ ]:
pred_times = []
for e in range(2):                                              # the 1st includes setup, so use the 2nd
    s = time.perf_counter()
    model.predict(val_ds.take(50), verbose=0)
    pred_times.append(time.perf_counter() - s)
print("predict times:", [round(t, 1) for t in pred_times])


predict times: [5.0, 3.4]


In [ ]:
train_steps = math.ceil(len(train) / 32)                        # 11,482
val_steps   = math.ceil(len(data["val"]) / 32)                  # 4,451

train_min = fit_times[1] / 50 * train_steps / 60
val_min   = pred_times[1] / 50 * val_steps / 60
epoch_min = train_min + val_min

print(f"Per step: {fit_times[1] / 50 * 1000:.0f} ms")
print(f"Train/epoch: {train_min:.0f} min | Val/epoch: {val_min:.0f} min | Total/epoch: {epoch_min:.0f} min")
print(f"3 epochs: {3 * epoch_min / 60:.1f} h")


Per step: 176 ms
Train/epoch: 34 min | Val/epoch: 5 min | Total/epoch: 39 min
3 epochs: 1.9 h


In [ ]:
def build_model():
    preprocessor = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
        "bert_small_en_uncased", sequence_length=512)
    model = keras_hub.models.BertTextClassifier.from_preset(
        "bert_small_en_uncased", num_classes=11, preprocessor=preprocessor)
    model.compile(
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=keras.optimizers.Adam(learning_rate=5e-5),
        metrics=["accuracy"],
    )
    return model

keras.utils.set_random_seed(42)
model = build_model()
model.evaluate(val_ds.take(1))                                  # back to ≈ 2.29 → fresh ✅


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.0000e+00 - loss: 2.4472


[2.447176218032837, 0.0]

In [ ]:
from src.callbacks import MacroF1

y_val = data["val"]["y"].values                     # same DataFrame as val_ds ✅

callbacks = [
    MacroF1(val_ds, y_val),                                              # 1. FIRST
    keras.callbacks.EarlyStopping(monitor="val_macro_f1", mode="max",
                                  patience=1, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(f"{ROOT}/models/bert_best.keras", monitor="val_macro_f1",
                                    mode="max", save_best_only=True),
    keras.callbacks.BackupAndRestore(backup_dir=f"{ROOT}/models/bert_backup"),
    keras.callbacks.CSVLogger(f"{ROOT}/models/bert_log.csv"),
]

start = time.perf_counter()
history = model.fit(train_ds, epochs=3, callbacks=callbacks)
print(f"Total time: {(time.perf_counter() - start) / 3600:.2f} h")


Epoch 2/3
11482/11482 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.9085 - loss: 0.2888Epoch 2: val macro-F1 = 0.7121
11482/11482 ━━━━━━━━━━━━━━━━━━━━ 2354s 203ms/step - accuracy: 0.9158 - loss: 0.2668 - val_macro_f1: 0.7121
Epoch 3/3
11482/11482 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.9123 - loss: 0.2728Epoch 3: val macro-F1 = 0.7205
11482/11482 ━━━━━━━━━━━━━━━━━━━━ 2342s 204ms/step - accuracy: 0.9136 - loss: 0.2682 - val_macro_f1: 0.7205
Total time: 1.31 h


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, f1_score

print(pd.read_csv(f"{ROOT}/models/bert_log.csv"))           # is epoch 0 (= epoch 1) there?

probs_bert = model.predict(val_ds, verbose=0)                 # best weights (epoch 3)
preds_bert = probs_bert.argmax(axis=1)
np.save(f"{ROOT}/models/bert_val_preds.npy", preds_bert)      # save for Day 13 comparisons 💾

print("macro-F1:", round(f1_score(y_val, preds_bert, average="macro"), 4))   # expect 0.7205
rep = pd.DataFrame(classification_report(y_val, preds_bert, target_names=label_order,
                                         output_dict=True)).T
print(rep.loc[label_order, ["support", "f1-score"]].round(3))


   epoch  accuracy      loss  val_macro_f1
0      1  0.915779  0.266784       0.71211
1      2  0.913634  0.268216       0.72052
macro-F1: 0.7205
                                                    support  f1-score
Checking or savings account                          9148.0     0.875
Credit card                                          3801.0     0.701
Credit reporting or other personal consumer rep...  91690.0     0.960
Debt collection                                      8124.0     0.674
Debt or credit management                             358.0     0.283
Money transfer, virtual currency, or money service  24817.0     0.952
Mortgage                                             1260.0     0.852
Payday loan, title loan, personal loan, or adva...    593.0     0.524
Prepaid card                                          402.0     0.641
Student loan                                         1201.0     0.802
Vehicle loan or lease                                1013.0     0.662


In [ ]:
import numpy

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip install -q keras-hub

import hashlib, tensorflow as tf, keras, keras_hub, pandas as pd
ROOT = "/content/drive/MyDrive/ticket_routing"

def sha256(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

print("bert_best.keras:", sha256(f"{ROOT}/models/bert_best.keras"))
print({"tensorflow": tf.__version__, "keras": keras.__version__,
       "keras_hub": keras_hub.__version__, "pandas": pd.__version__})


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bert_best.keras: 58dab6bbc8b8dcbd469243825eb59bf77cbe1882ac7c7d941d02e8a91d4bd66a
{'tensorflow': '2.21.0', 'keras': '3.13.2', 'keras_hub': '0.26.0', 'pandas': '2.2.3'}


In [ ]:
print("Drive text.py:", sha256(f"{ROOT}/src/text.py")[:12], "| expect 26f00e8b982d")

import sys, json
sys.path.append(ROOT)
from src.text import clean_text, format_for_bert

label_order = json.load(open(f"{ROOT}/models/label_order.json"))
label_to_id = {l: i for i, l in enumerate(label_order)}

def first_350(text):
    return " ".join(text.split()[:350])

bert_ds, y_test = {}, {}
for name in ["test", "test_dedup"]:
    df = pd.read_parquet(f"{ROOT}/data/processed/{name}.parquet")
    print(name, df.shape, "| NaN labels:", df["label"].map(label_to_id).isna().sum())
    y_test[name] = df["label"].map(label_to_id).values
    texts = df["Consumer complaint narrative"].map(clean_text).map(first_350).map(format_for_bert)
    bert_ds[name] = tf.data.Dataset.from_tensor_slices(texts.values).batch(32).prefetch(tf.data.AUTOTUNE)   # no shuffle


Drive text.py: 26f00e8b982d | expect 26f00e8b982d


ModuleNotFoundError: No module named 'src'

In [ ]:
import os, sys, json, hashlib, importlib.util
import numpy as np, pandas as pd, tensorflow as tf

ROOT = "/content/drive/MyDrive/ticket_routing"

def sha256(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

# 🔏 seal check
print("Drive text.py:", sha256(f"{ROOT}/src/text.py")[:12], "| expect 26f00e8b982d")

# load the frozen text.py directly by its path (avoids the 'No module named src' issue)
spec = importlib.util.spec_from_file_location("text", f"{ROOT}/src/text.py")
text_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(text_mod)
clean_text, format_for_bert = text_mod.clean_text, text_mod.format_for_bert

label_order = json.load(open(f"{ROOT}/models/label_order.json"))
label_to_id = {l: i for i, l in enumerate(label_order)}

def first_350(text):
    return " ".join(text.split()[:350])

bert_ds, y_test = {}, {}
for name in ["test", "test_dedup"]:
    df = pd.read_parquet(f"{ROOT}/data/processed/{name}.parquet")
    y = df["label"].map(label_to_id)
    print(name, df.shape, "| NaN labels:", y.isna().sum())
    y_test[name] = y.values
    texts = df["Consumer complaint narrative"].map(clean_text).map(first_350).map(format_for_bert)
    bert_ds[name] = (tf.data.Dataset.from_tensor_slices(texts.values)
                     .batch(32).prefetch(tf.data.AUTOTUNE))          # no shuffle

print("BERT test inputs ready ✅ (no predictions yet)")


Drive text.py: 26f00e8b982d | expect 26f00e8b982d
test (169189, 5) | NaN labels: 0
test_dedup (87091, 5) | NaN labels: 0
BERT test inputs ready ✅ (no predictions yet)


In [ ]:
import keras, keras_hub
from scipy.special import softmax

os.makedirs(f"{ROOT}/predictions", exist_ok=True)
bert = keras.models.load_model(f"{ROOT}/models/bert_best.keras")

for s in ["test", "test_dedup"]:
    logits = bert.predict(bert_ds[s], verbose=0).astype("float32")   # float16 → float32 FIRST
    proba = softmax(logits, axis=1)                                     # logits → probabilities
    pred = proba.argmax(axis=1)
    assert len(pred) == len(y_test[s]), "Row count mismatch!"
    assert np.allclose(proba.sum(axis=1), 1, atol=1e-3), "Rows don't sum to 1!"
    assert (proba.argmax(axis=1) == pred).all()
    np.save(f"{ROOT}/predictions/proba_bert_{s}.npy", proba)
    np.save(f"{ROOT}/predictions/pred_bert_{s}.npy", pred)
    print(f"✅ bert    {s:10} proba {proba.shape} pred {pred.shape}")


/usr/local/lib/python3.13/dist-packages/keras/src/saving/serialization_lib.py:749: UserWarning: `compile()` was not called as part of model loading because the model's `compile()` method is custom. All subclassed Models that have `compile()` overridden should also override `get_compile_config()` and `compile_from_config(config)`. Alternatively, you can call `compile()` manually after loading.
  instance.compile_from_config(compile_config)
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 4 variables whereas the saved optimizer has 152 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 148 variables. 
  saveable.load_own_variables(weights_store.get(inner_pa

✅ bert    test       proba (169189, 11) pred (169189,)
✅ bert    test_dedup proba (87091, 11) pred (87091,)
